In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("rlcard") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rlcard"])

In [2]:
from games.leduc import Leduc
from agents.counterfactualregret_t import CounterFactualRegret
from agents.agent_random import RandomAgent

import numpy as np
from collections import defaultdict
from tqdm import tqdm

In [170]:
game = Leduc(render_mode="human")
game.reset(seed=1)

print("Agents:", game.agents)
print("Current agent:", game.agent_selection)
print("Observation:", game.observe(game.agent_selection))
print("Legal actions:", game.available_actions())

Agents: ['player_0', 'player_1']
Current agent: player_0
Observation: K_#_1_2_0
Legal actions: [0, 1, 2]


In [171]:
agents_random = {
    agent: RandomAgent(game=game, agent=agent)
    for agent in game.agents
}
agents_random

{'player_0': <agents.agent_random.RandomAgent at 0x122e65e3d50>,
 'player_1': <agents.agent_random.RandomAgent at 0x122e40f2d10>}

In [177]:
game.reset(seed=1)

while not game.game_over():
    agent = game.agent_selection
    legal_actions = game.available_actions()
    action = agents_random[agent].action()

    game.render()
    print(f"Agent {agent}")
    print(f"Observation: {game.observe(agent)}")
    print(f"Legal actions: {legal_actions}")
    print(f"Action {action} - move {game._moves[action]}")
    print()

    game.step(action)

game.render()
for agent in game.agents:
    print(f"Reward {agent} = {game.reward(agent)}")

Agent player_0
Observation: K_#_1_2_0
Legal actions: [0, 1, 2]
Action 1 - move r

Agent player_1
Observation: J_#_4_2_0r
Legal actions: [0, 1, 2]
Action 2 - move f

Reward player_0 = 1.0
Reward player_1 = -1.0


In [178]:
game = Leduc(render_mode="")
game.reset(seed=1)

agents_cfr = [
    CounterFactualRegret(game=game, agent=game.agents[0]),
    CounterFactualRegret(game=game, agent=game.agents[1])
]
agents_cfr

In [229]:
n_training_iterations = 100
    
for i, agent in enumerate(game.agents):
    print(f"Training agent {agent}")
    agents_cfr[i].train(n_training_iterations)
    print(f"Known information sets: {len(agents_cfr[i].node_dict)}")

Training agent player_0


100%|██████████| 100/100 [00:33<00:00,  3.02it/s]


Known information sets: 576
Training agent player_1


100%|██████████| 100/100 [00:33<00:00,  2.97it/s]

Known information sets: 576


In [235]:
for i, agent in enumerate(game.agents):
    print(f"First 10 policies for {agent} (total information sets: {len(agents_cfr[i].node_dict)})")
    for obs, node in list(agents_cfr[i].node_dict.items())[:10]:
        policy = [(move, round(float(prob), 3)) for move, prob in zip(game._moves, node.policy(node.game.available_actions()))]
        print(obs, policy)
    print()

First 10 policies for player_0 (total information sets: 576)
K_#_1_2_0 [('c', 0.126), ('r', 0.866), ('f', 0.008), ('k', 0.0)]
Q_#_2_2_0c [('c', 0.0), ('r', 0.512), ('f', 0.013), ('k', 0.475)]
K_#_2_4_0cr [('c', 0.424), ('r', 0.569), ('f', 0.007), ('k', 0.0)]
Q_K_4_4_0crc [('c', 0.0), ('r', 0.54), ('f', 0.017), ('k', 0.443)]
K_K_4_8_0crcr [('c', 0.073), ('r', 0.899), ('f', 0.029), ('k', 0.0)]
Q_K_12_8_0crcrr [('c', 0.983), ('r', 0.0), ('f', 0.017), ('k', 0.0)]
K_K_4_4_0crck [('c', 0.0), ('r', 0.893), ('f', 0.029), ('k', 0.079)]
Q_K_8_4_0crckr [('c', 0.375), ('r', 0.608), ('f', 0.017), ('k', 0.0)]
K_K_8_12_0crckrr [('c', 0.971), ('r', 0.0), ('f', 0.029), ('k', 0.0)]
Q_#_6_4_0crr [('c', 0.992), ('r', 0.0), ('f', 0.008), ('k', 0.0)]

First 10 policies for player_1 (total information sets: 576)
Q_#_2_1_1 [('c', 0.401), ('r', 0.566), ('f', 0.033), ('k', 0.0)]
J_#_2_2_1c [('c', 0.0), ('r', 0.224), ('f', 0.124), ('k', 0.652)]
Q_#_4_2_1cr [('c', 0.407), ('r', 0.58), ('f', 0.014), ('k', 0.0)]
J_

In [247]:
cum_rewards = defaultdict(float)
n_games = 1

for _ in tqdm(range(n_games)):
    game.reset()
    while not game.game_over():
        agent = game.agent_selection
        agent_index = game.agents.index(agent)
        action = agents_cfr[agent_index].action()
        obs = game.observe(agent)
        print(f"Agent {agent} - Observation: {obs} - Action: {game._moves[action]}") if n_games == 1 else None
        game.step(action)

    for agent in game.agents:
        cum_rewards[agent] += game.reward(agent)

print("Average rewards:")
for agent in game.agents:
    print(f"{agent}: {cum_rewards[agent] / n_games:.3f}")

100%|██████████| 1/1 [00:00<00:00, 1006.79it/s]

Agent player_1 - Observation: Q_#_2_1_1 - Action: c
Agent player_0 - Observation: J_#_2_2_1c - Action: r
Agent player_1 - Observation: Q_#_4_2_1cr - Action: c
Agent player_0 - Observation: J_J_4_4_1crc - Action: k
Agent player_1 - Observation: Q_J_4_4_1crck - Action: k
Average rewards:
player_0: 2.000
player_1: -2.000
